# Notebook 01 - EDA
## Projet M3 - Maintenance predictive d'equipements miniers

Ce notebook realise une analyse exploratoire complete sur les donnees Azure Predictive Maintenance, avec integration du jeu AI4I 2020 comme reference complementaire. L'objectif est de caracteriser les capteurs, le desequilibre de classes, les pannes et les tendances temporelles avant toute modelisation.


## 1. Constantes et chargement

Toutes les constantes sont explicites pour garantir la reproductibilite. Aucun `random split` n'est utilise dans ce projet afin d'eviter toute fuite temporelle.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import prepare_raw_datasets
from src.feature_engineering import build_feature_table, temporal_train_test_split, get_feature_columns
from src.project_pipeline import run_full_training
from src.models import train_classifiers, train_rul_models
from src.evaluation import evaluate_classifier, evaluate_regressor
from src.utils import COLOR_PALETTE, RANDOM_STATE, SENSOR_COLUMNS, CRITICAL_RUL_THRESHOLD

plt.style.use("seaborn-v0_8")
sns.set_palette([COLOR_PALETTE["accent"], COLOR_PALETTE["success"], COLOR_PALETTE["info"], COLOR_PALETTE["danger"]])
pd.set_option("display.max_columns", 120)


In [ ]:
# Constantes de visualisation et de controle
TOP_MACHINES = 5
SAMPLE_MACHINES = [1, 2, 3, 4, 5]

bundle = prepare_raw_datasets(PROJECT_ROOT)
features = build_feature_table(bundle, PROJECT_ROOT)
train_df, test_df = temporal_train_test_split(features)

# ... Nettoyage : prints supprimés ...
features.head()


## 2. Statistiques descriptives

Les statistiques suivantes couvrent les variables capteurs principales et mettent en evidence les echelles de variation par machine.


In [ ]:
descriptive_stats = features[SENSOR_COLUMNS + ["age", "error_count", "error_count_cumulative", "rul_hours"]].describe().T
descriptive_stats


## 3. Distribution des capteurs

On verifie les distributions de `volt`, `rotate`, `pressure` et `vibration` pour identifier les asymetries, dispersions et eventuelles valeurs aberrantes.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, sensor in zip(axes.ravel(), SENSOR_COLUMNS):
    sns.histplot(features[sensor], kde=True, ax=ax, color=COLOR_PALETTE["accent"])
    ax.set_title(f"Distribution de {sensor}")
    ax.set_xlabel(sensor)
    ax.set_ylabel("Frequence")
plt.tight_layout()
plt.show()


## 4. Frequence des pannes

Cette vue compare la frequence des pannes par composant et par machine, ce qui aide a justifier la difficulte du probleme de classification.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
bundle.failures["failure"].value_counts().plot(kind="bar", ax=axes[0], color=COLOR_PALETTE["danger"])
axes[0].set_title("Frequence des pannes par type")
axes[0].set_xlabel("Type de panne")
axes[0].set_ylabel("Nombre d'evenements")

bundle.failures["machineID"].value_counts().head(15).sort_values().plot(kind="barh", ax=axes[1], color=COLOR_PALETTE["info"])
axes[1].set_title("Top 15 machines les plus en panne")
axes[1].set_xlabel("Nombre de pannes")
axes[1].set_ylabel("Machine")
plt.tight_layout()
plt.show()


## 5. Correlations

La heatmap annotee permet d'identifier les associations lineaires utiles pour le feature engineering et d'anticiper les colinearites.


In [ ]:
corr_cols = SENSOR_COLUMNS + ["age", "error_count_cumulative", "hours_since_last_failure", "rul_hours", "failure_within_24h"]
corr = features[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Heatmap de correlations")
plt.tight_layout()
plt.show()


## 6. Degradation temporelle

Nous visualisons 5 machines exemple sur la serie temporelle pour observer les signaux de degradation avant panne.


In [ ]:
subset = features[features["machineID"].isin(SAMPLE_MACHINES)].copy()
fig, axes = plt.subplots(len(SAMPLE_MACHINES), 1, figsize=(16, 14), sharex=True)
for ax, machine_id in zip(axes, SAMPLE_MACHINES):
    machine_slice = subset[subset["machineID"] == machine_id].tail(24 * 14)
    ax.plot(machine_slice["datetime"], machine_slice["vibration"], label="Vibration", color=COLOR_PALETTE["danger"])
    ax.plot(machine_slice["datetime"], machine_slice["pressure"], label="Pressure", color=COLOR_PALETTE["info"], alpha=0.8)
    ax.set_title(f"Machine {machine_id} - degradation sur 14 jours")
    ax.set_ylabel("Valeur capteur")
    ax.legend()
axes[-1].set_xlabel("Temps")
plt.tight_layout()
plt.show()


## 7. Desequilibre de classes

La variable cible est definie comme une panne dans les 24h. Le ratio ci-dessous justifie l'usage de `class_weight` et le test de `SMOTE`.


In [ ]:
class_ratio = features["failure_within_24h"].value_counts(normalize=True).rename({0: "Normal", 1: "Panne <24h"})
# ... Nettoyage : prints supprimés ...
plt.figure(figsize=(6, 4))
class_ratio.plot(kind="bar", color=[COLOR_PALETTE["success"], COLOR_PALETTE["danger"]])
plt.title("Distribution de la cible")
plt.ylabel("Proportion")
plt.tight_layout()
plt.show()


## 8. Valeurs manquantes et aberrantes

Cette section verifie la qualite des donnees et permet d'expliquer les decisions de pretraitement.


In [ ]:
missing = features.isna().mean().sort_values(ascending=False).head(15)
missing


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, sensor in zip(axes.ravel(), SENSOR_COLUMNS):
    sns.boxplot(data=features, y=sensor, ax=ax, color=COLOR_PALETTE["accent"])
    ax.set_title(f"Boxplot - {sensor}")
    ax.set_ylabel(sensor)
plt.tight_layout()
plt.show()


## 9. Synthese EDA

Les capteurs montrent des regimes de fonctionnement distincts selon les machines, les pannes restent rares mais observables, et les variables temporelles seront essentielles pour capturer la degradation. Le split chronologique est deja prepare pour garantir une evaluation honnete.
